# Lesson 20 Exercise

Importing the required libraries

In [1]:
import pandas as pd
import nltk
import time
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [2]:
nltk.download('stopwords')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
df = pd.read_csv('../data/Hotel_Reviews.csv')
print(df.shape)

(515738, 17)


In [4]:
print(df.columns.tolist())

['Hotel_Address', 'Additional_Number_of_Scoring', 'Review_Date', 'Average_Score', 'Hotel_Name', 'Reviewer_Nationality', 'Negative_Review', 'Review_Total_Negative_Word_Counts', 'Total_Number_of_Reviews', 'Positive_Review', 'Review_Total_Positive_Word_Counts', 'Total_Number_of_Reviews_Reviewer_Has_Given', 'Reviewer_Score', 'Tags', 'days_since_review', 'lat', 'lng']


In [5]:
df.drop(["Additional_Number_of_Scoring", "Review_Total_Negative_Word_Counts", "Review_Total_Positive_Word_Counts", "days_since_review", "lat", "lng"], axis=1, inplace=True)
print(df.columns.tolist())

['Hotel_Address', 'Review_Date', 'Average_Score', 'Hotel_Name', 'Reviewer_Nationality', 'Negative_Review', 'Total_Number_of_Reviews', 'Positive_Review', 'Total_Number_of_Reviews_Reviewer_Has_Given', 'Reviewer_Score', 'Tags']


In [6]:
def replace_address(row):
    if "Netherlands" in row["Hotel_Address"]:
        return "Amsterdam, Netherlands"
    elif "Barcelona" in row["Hotel_Address"]:
        return "Barcelona, Spain"
    elif "United Kingdom" in row["Hotel_Address"]:
        return "London, United Kingdom"
    elif "Milan" in row["Hotel_Address"]:
        return "Milan, Italy"
    elif "France" in row["Hotel_Address"]:
        return "Paris, France"
    elif "Vienna" in row["Hotel_Address"]:
        return "Vienna, Austria"

df["Hotel_Address"]= df.apply(replace_address, axis=1)
print(df["Hotel_Address"].value_counts())

Hotel_Address
London, United Kingdom    262301
Barcelona, Spain           60149
Paris, France              59928
Amsterdam, Netherlands     57214
Vienna, Austria            38939
Milan, Italy               37207
Name: count, dtype: int64


In [7]:
df.Total_Number_of_Reviews = df.groupby("Hotel_Name").Hotel_Name.transform("count")
df.Average_Score = round(df.groupby("Hotel_Name").Reviewer_Score.transform("mean"), 1)
print(df[["Hotel_Name", "Total_Number_of_Reviews", "Average_Score"]].head(10))

    Hotel_Name  Total_Number_of_Reviews  Average_Score
0  Hotel Arena                      405            7.8
1  Hotel Arena                      405            7.8
2  Hotel Arena                      405            7.8
3  Hotel Arena                      405            7.8
4  Hotel Arena                      405            7.8
5  Hotel Arena                      405            7.8
6  Hotel Arena                      405            7.8
7  Hotel Arena                      405            7.8
8  Hotel Arena                      405            7.8
9  Hotel Arena                      405            7.8


In [9]:
df.Tags = df.Tags.str.strip("[]")
df.Tags = df.Tags.str.replace(" ' ", ",", regex=False)
df.Tags = df.Tags.str.replace("' ", "", regex=False)
df.Tags = df.Tags.str.replace(" '", "", regex=False)
print(df.Tags.head(5))

0    Leisure trip,,Couple,,Duplex Double Room,,Stay...
1    Leisure trip,,Couple,,Duplex Double Room,,Stay...
2    Leisure trip,,Family with young children,,Dupl...
3    Leisure trip,,Solo traveler,,Duplex Double Roo...
4    Leisure trip,,Couple,,Suite,,Stayed 2 nights,,...
Name: Tags, dtype: str


In [10]:
df["Leisure_trip"] = df.Tags.apply(lambda tag: 1 if "Leisure trip" in tag else 0)
df["Couple"]= df.Tags.apply(lambda tag: 1 if "Couple" in tag else 0)
df["Solo_traveler"] = df.Tags.apply(lambda tag: 1 if "Solo traveler" in tag else 0)
df["Business_trip"] = df.Tags.apply(lambda tag: 1 if "Business trip" in tag else 0)
df["Family"] = df.Tags.apply(lambda tag: 1 if "Family" in tag else 0)
print(df[["Tags", "Leisure_trip", "Couple", "Solo_traveler", "Business_trip", "Family"]].head(5))

                                                Tags  Leisure_trip  Couple  \
0  Leisure trip,,Couple,,Duplex Double Room,,Stay...             1       1   
1  Leisure trip,,Couple,,Duplex Double Room,,Stay...             1       1   
2  Leisure trip,,Family with young children,,Dupl...             1       0   
3  Leisure trip,,Solo traveler,,Duplex Double Roo...             1       0   
4  Leisure trip,,Couple,,Suite,,Stayed 2 nights,,...             1       1   

   Solo_traveler  Business_trip  Family  
0              0              0       0  
1              0              0       0  
2              0              0       1  
3              1              0       0  
4              0              0       0  


In [11]:
start = time.time()
cache = set(stopwords.words("english"))

def remove_stopwords(review):
    text = " ".join([word for word in review.split() if word not in cache])
    return text

df.Negative_Review = df.Negative_Review.apply(remove_stopwords)
df.Positive_Review = df.Positive_Review.apply(remove_stopwords)

end = time.time()
print("Stop words removed in", round(end - start, 2), "seconds")

Stop words removed in 2.64 seconds


In [12]:
vader_sentiment = SentimentIntensityAnalyzer()

def calc_sentiment(review):
    if review == "No Negative" or review == "No Positive":
        return 0
    return vader_sentiment.polarity_scores(review)["compound"]

start = time.time()
df["Negative_Sentiment"] = df.Negative_Review.apply(calc_sentiment)
df["Positive_Sentiment"] = df.Positive_Review.apply(calc_sentiment)
end = time.time()
print("Sentiment calculated in", round(end - start, 2), "seconds")

Sentiment calculated in 103.0 seconds


In [13]:
vader_sentiment = SentimentIntensityAnalyzer()

def calc_sentiment(review):
    if review == "No Negative" or review == "No Positive":
        return 0
    return vader_sentiment.polarity_scores(review)["compound"]

start = time.time()
df["Negative_Sentiment"] = df.Negative_Review.apply(calc_sentiment)
df["Positive_Sentiment"] = df.Positive_Review.apply(calc_sentiment)
end = time.time()
print("Sentiment calculated in", round(end - start, 2), "seconds")

Sentiment calculated in 102.9 seconds


In [14]:
df = df.sort_values(by=["Negative_Sentiment"], ascending=True)
print(df[["Negative_Review", "Negative_Sentiment"]].head(3))

                                          Negative_Review  Negative_Sentiment
186584  So bad experience memories I hotel The first n...             -0.9920
129503  First charged twice room booked booking second...             -0.9896
307286  The staff Had bad experience even booking Janu...             -0.9889


In [15]:
df = df.reindex(["Hotel_Name", "Hotel_Address", "Total_Number_of_Reviews", "Average_Score", "Reviewer_Score", "Negative_Review", "Positive_Review", "Negative_Sentiment", "Positive_Sentiment", "Reviewer_Nationality", "Leisure_trip", "Couple", "Solo_traveler", "Business_trip", "Family", "Tags"], axis=1)

df.to_csv('../data/Hotel_Reviews_NLP.csv', index=False)
print("File saved successfully")

File saved successfully
